# Progressive Quantization Curriculum Demo

This notebook demonstrates the progressive quantization curriculum:
1. How the scheduler transitions between phases
2. How weight distributions change at each phase
3. How the model adapts to lower precision

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
import numpy as np

from modeling.config import EdgeBitConfig
from modeling.model import EdgeBitForCausalLM
from modeling.bitlinear import BitLinear
from training.quant_scheduler import QuantizationScheduler

## 1. Curriculum Schedule Visualization

In [ ]:
config = EdgeBitConfig.tiny()
config.vocab_size = 500
model = EdgeBitForCausalLM(config)
total_steps = 1000

scheduler = QuantizationScheduler.from_total_steps(total_steps, model)
print(scheduler.summary())

# Visualize phases
phases = scheduler.phases
colors = {'none': '#2563EB', 'int8': '#059669', 'int4': '#F59E0B', 'ternary': '#DC2626'}

fig, ax = plt.subplots(figsize=(12, 2))
for p in phases:
    ax.barh(0, p.end_step - p.start_step, left=p.start_step, height=0.5,
            color=colors.get(p.quant_mode, '#6B7280'), label=f'{p.name} ({p.quant_mode})')
    ax.text((p.start_step + p.end_step) / 2, 0, f'{p.name}\n{p.quant_mode}',
            ha='center', va='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Training Step')
ax.set_title('Progressive Quantization Curriculum')
ax.set_yticks([])
ax.set_xlim(0, total_steps)
plt.tight_layout()
plt.show()

## 2. Weight Distribution Through Phases

Simulate how weight distributions evolve during curriculum training.

In [ ]:
from modeling.quant_utils import ternary_quantize_absmean, int8_symmetric_quantize

# Start with random weights
w = torch.randn(256, 256) * 0.02

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Phase 1: BF16 (no quantization)
axes[0].hist(w.flatten().numpy(), bins=80, color='#2563EB', alpha=0.7)
axes[0].set_title('Phase 1: BF16 (Float)')
axes[0].set_xlabel('Weight Value')

# Phase 2: INT8
w_int8, scale = int8_symmetric_quantize(w.unsqueeze(0))
w_recon8 = w_int8.float().squeeze(0) * scale
axes[1].hist(w_recon8.flatten().numpy(), bins=80, color='#059669', alpha=0.7)
axes[1].set_title('Phase 2: INT8')
axes[1].set_xlabel('Weight Value')

# Phase 3: INT4 (simulate with fewer levels)
scale4 = w.abs().max() / 7
w_int4 = (w / scale4).round().clamp(-8, 7) * scale4
axes[2].hist(w_int4.flatten().numpy(), bins=80, color='#F59E0B', alpha=0.7)
axes[2].set_title('Phase 3: INT4')
axes[2].set_xlabel('Weight Value')

# Phase 4: Ternary
w_ternary, scale_t = ternary_quantize_absmean(w, group_size=128)
axes[3].hist(w_ternary.flatten().numpy(), bins=5, color='#DC2626', alpha=0.7, rwidth=0.5)
axes[3].set_title('Phase 4: Ternary')
axes[3].set_xlabel('Weight Value')
axes[3].set_xticks([-1, 0, 1])

plt.suptitle('Weight Distribution Through Quantization Phases', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

# Statistics
print(f"Ternary weight distribution:")
print(f"  Zero:  {(w_ternary == 0).float().mean()*100:.1f}%")
print(f"  +1:    {(w_ternary == 1).float().mean()*100:.1f}%")
print(f"  -1:    {(w_ternary == -1).float().mean()*100:.1f}%")

## 3. Simulated Loss Curve

What a typical loss curve looks like during progressive quantization.

In [ ]:
# Simulated loss curve showing phase transitions
np.random.seed(42)
steps = np.arange(1000)

# Base loss curve (exponential decay)
loss = 10 * np.exp(-steps / 300) + 3

# Add phase transition spikes
transitions = [100, 300, 550]  # BF16->INT8, INT8->INT4, INT4->Ternary
spike_sizes = [0.8, 1.5, 3.0]

for t, spike in zip(transitions, spike_sizes):
    decay = np.exp(-(steps - t) / 50)
    decay[steps < t] = 0
    loss += spike * decay

# Add noise
loss += np.random.normal(0, 0.15, len(steps))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(steps, loss, color='#2563EB', linewidth=0.8, alpha=0.8)

# Phase regions
phase_info = [
    (0, 100, 'BF16\nWarmup', '#2563EB'),
    (100, 300, 'INT8\nAdapt', '#059669'),
    (300, 550, 'INT4\nAdapt', '#F59E0B'),
    (550, 1000, 'Ternary\nFinal', '#DC2626'),
]
for start, end, label, color in phase_info:
    ax.axvspan(start, end, alpha=0.1, color=color)
    ax.text((start + end) / 2, ax.get_ylim()[1] * 0.95, label,
            ha='center', va='top', fontsize=9, color=color, fontweight='bold')

for t in transitions:
    ax.axvline(t, color='gray', linestyle='--', alpha=0.5)

ax.set_xlabel('Training Step')
ax.set_ylabel('Loss')
ax.set_title('Simulated Training Loss with Progressive Quantization Curriculum')
ax.set_ylim(2, 14)
plt.tight_layout()
plt.show()

## Key Observations

1. **Phase transitions cause loss spikes** — this is expected and recoverable
2. **The ternary spike is the largest** — going from 16 levels to 3 is the biggest jump
3. **Recovery takes 100-500 steps** — the model adapts to each precision level
4. **Final ternary loss is higher than BF16 minimum** — this is the quantization cost
5. **The curriculum makes ternary training possible** — direct ternary training diverges